**Teste de estrutura de dados**

In [ ]:
import numpy as np
import pandas as pd

df: pd.DataFrame = pd.read_csv(  # pyright: ignore[reportUnknownMemberType]
    "../../src/estudo_twins_digital/app/data/raw/dados_brutos_motores.csv"
)

Calculando a Vida Útil Remanescente (RUL)

In [ ]:
vida_util_total = (
    df.groupby("motor_id")["horas_operacao"]
    .max()
    .to_frame("vida_util_total")
    .reset_index()
)
df_estruturado = pd.merge(df, vida_util_total, on="motor_id")
df_estruturado["RUL"] = (
    df_estruturado["vida_util_total"] - df_estruturado["horas_operacao"]
)
df_estruturado.drop(columns=["vida_util_total"])

,motor_id,horas_operacao,temperatura_c,vibracao_hz,rotacao_rpm,corrente_a,falha_registrada,RUL
0,motor_01,1,69.29,9.71,1806.74,14.92,0,264
1,motor_01,2,69.70,10.26,1801.98,14.93,0,263
2,motor_01,3,72.12,11.31,1806.04,15.11,0,262
3,motor_01,4,68.59,10.28,1809.14,15.13,0,261
4,motor_01,5,70.93,9.90,1802.19,15.19,0,260
...,...,...,...,...,...,...,...,...
3526,motor_15,283,99.51,25.65,1754.05,17.91,0,4
3527,motor_15,284,98.48,24.49,1745.32,17.88,0,3
3528,motor_15,285,100.53,23.76,1743.56,18.26,0,2
3529,motor_15,286,99.38,25.66,1745.54,17.82,0,1


Criando atributos media_movel e desvio_padrao com janela deslizante

In [ ]:
tamanho_janela: int = 5
sensores: list[str] = [
    "temperatura_c",
    "vibracao_hz",
    "rotacao_rpm",
    "corrente_a",
]
for sensor in sensores:
    # Calcula a média móvel (tendência central recente)
    df_estruturado[f"{sensor}_media_movel"] = df_estruturado.groupby(
        "motor_id"
    )[sensor].transform(
        lambda x: x.rolling(window=tamanho_janela, min_periods=1).mean()
    )
    # Calcula o desvio padrão móvel (volatilidade ou instabilidade recente)
    df_estruturado[f"{sensor}_desvio_padrao"] = df_estruturado.groupby(
        "motor_id"
    )[sensor].transform(
        lambda x: x.rolling(window=tamanho_janela, min_periods=1).std()
    )
df_estruturado

,motor_id,horas_operacao,temperatura_c,vibracao_hz,rotacao_rpm,corrente_a,falha_registrada,vida_util_total,RUL,temperatura_c_media_movel,temperatura_c_desvio_padrao,vibracao_hz_media_movel,vibracao_hz_desvio_padrao,rotacao_rpm_media_movel,rotacao_rpm_desvio_padrao,corrente_a_media_movel,corrente_a_desvio_padrao
0,motor_01,1,69.29,9.71,1806.74,14.92,0,265,264,69.290,NaN,9.710000,NaN,1806.740,NaN,14.920000,NaN
1,motor_01,2,69.70,10.26,1801.98,14.93,0,265,263,69.495,0.289914,9.985000,0.388909,1804.360,3.365828,14.925000,0.007071
2,motor_01,3,72.12,11.31,1806.04,15.11,0,265,262,70.370,1.529346,10.426667,0.812917,1804.920,2.570058,14.986667,0.106927
3,motor_01,4,68.59,10.28,1809.14,15.13,0,265,261,69.925,1.533417,10.390000,0.667782,1805.975,2.975830,15.022500,0.112953
4,motor_01,5,70.93,9.90,1802.19,15.19,0,265,260,70.126,1.401974,10.292000,0.618442,1805.218,3.083329,15.056000,0.123207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3526,motor_15,283,99.51,25.65,1754.05,17.91,0,287,4,98.648,1.172762,24.318000,0.865604,1755.098,5.937985,17.840000,0.098489
3527,motor_15,284,98.48,24.49,1745.32,17.88,0,287,3,98.326,0.856143,24.460000,0.811850,1753.150,7.376873,17.864000,0.088204
3528,motor_15,285,100.53,23.76,1743.56,18.26,0,287,2,99.004,1.010510,24.496000,0.765755,1751.400,8.567716,17.974000,0.161028
3529,motor_15,286,99.38,25.66,1745.54,17.82,0,287,1,99.188,0.969624,24.858000,0.810537,1750.680,8.947947,17.952000,0.175414
